# 运行 3 个训练 step

本节使用默认的两卡 FSDP2 + TND 配置执行 TorchTitan-NPU Wordle 训练。模型、数据、train batch size、prompt/response 长度、rollout 数量、六轮 AgentLoop、GRPO 和奖励函数均沿用初阶配置，只把总训练步数限制为 3。

一次 step 会依次经过 vLLM rollout、Wordle 交互与奖励、GRPO、Actor 更新和权重同步。连续完成 3 个 step 后，就可以沿日志确认 TorchTitan Engine、FSDP2、offload 和 TND 已经接入这条 RL 流水线。三步日志也会给出 loss 和阶段耗时，但样本量还不足以观察收敛趋势。


## 运行下面的 Cell

Cell 会以 DP shard 2、CP 1 连续运行 3 个训练 step。初始化阶段应出现 TorchTitan FSDP2 和 NPU varlen attention 日志；训练开始后，可以依次看到 rollout、奖励、log-prob、Actor update 和权重同步相关输出。进度到达 3/3，并输出 Actor loss、梯度或学习率等指标，说明三次参数更新已经完成。

如果中途失败，先找到对应 worker 的第一条异常，再根据最后完成的阶段判断问题位于 rollout、Wordle 交互、训练计算还是权重同步。


In [ ]:
%%bash
set -euo pipefail

COURSE_ROOT=$(git rev-parse --show-toplevel)
TRAIN_DIR="$(dirname "${COURSE_ROOT}")/cann-recipes-train/llm_rl/qwen3_wordle"
cd "${TRAIN_DIR}"
TOTAL_TRAINING_STEPS=3 \
SAVE_FREQ=-1 \
TEST_FREQ=-1 \
    bash torchtitan_backend/run_qwen3_1.7b_wordle_torchtitan_npu.sh


`SAVE_FREQ=-1` 与 `TEST_FREQ=-1` 分别跳过 checkpoint 保存和训练结束后的验证，训前验证仍按 launcher 默认值执行。这样可以把时间集中在三次完整的 RL 更新上。准备长期训练时，再根据任务需要设置保存频率、验证频率和恢复方式。


## 课后练习

### 判断题

1. （判断题）三步训练保持初阶 batch、长度、rollout 数量和 AgentLoop 轮数不变。

2. （判断题）完成 3 个 step 可以证明切换训练后端后一定获得性能提升。

### 单选题

3. （单选题）本节限制训练步数的唯一训练规模覆盖项是什么？

   A. `TOTAL_TRAINING_STEPS=3`

   B. `TRAIN_BATCH_SIZE=2`

   C. `MAX_RESPONSE_LENGTH=512`

   D. `ROLLOUT_N=2`

### 多选题

4. （多选题）一个完整训练 step 应经过哪些关键阶段？

   A. vLLM rollout

   B. 奖励与 GRPO 优势

   C. Actor 前反向和优化器更新

   D. Actor 权重同步到 vLLM

5. （多选题）三步训练可以支持哪些结论？

   A. TorchTitan Engine 能完成 Actor 更新

   B. 原 Wordle RL 数据流可以运行

   C. FSDP2、TND 与权重同步路径可以执行

   D. TorchTitan 一定比 FSDP 更快


> 完成练习后，运行下方单元格查看参考答案和解析。


In [ ]:
from pathlib import Path
import subprocess

course_root = Path(subprocess.check_output(['git', 'rev-parse', '--show-toplevel'], text=True).strip())
answer_path = course_root / 'tutorials/rl_training_pipeline/07_torchtitan_wordle_training/answer/07.04_answer.txt'
assert answer_path.is_file(), f'未找到答案文件: {answer_path}'
print(answer_path.read_text(encoding='utf-8'))
